# Citi Bike Feature Engineering

## Project Objective

The objective of this project is to develop a machine learning dataset to predict whether a Citi Bike station will experience a bike shortage or dock shortage within the next 60 minutes.

## Data Sources

The project uses three datasets:

1. **Station Information:** Station identifiers, geographic locations, and station capacities.
2. **Station Status History:** Historical bike availability, dock availability, and observation timestamps.
3. **Weather Forecast History:** Temperature, precipitation, and wind speed.

## Feature Engineering Objectives

- Clean and validate historical station availability data.
- Generate time-based features, including hour of day and day of week.
- Calculate station occupancy and availability ratios.
- Incorporate relevant weather information.
- Generate prediction labels for bike and dock shortages within the next 60 minutes.
- Prepare a structured dataset for machine learning.

## Data Quality and Modeling Considerations

- Use actual observation timestamps to generate prediction labels.
- Identify stale observations and account for data collection gaps.
- Prevent data leakage by using only information available at prediction time.

In [0]:

# Load the existing Delta tables from the Databricks catalog
df_station_info = spark.table("workspace.citibike.station_information")
df_status = spark.table("workspace.citibike.station_status_history")
df_weather = spark.table("workspace.citibike.weather_forecast_history")

# Verify that each dataset is accessible and display its record count
print("Station information records:", df_station_info.count())
print("Station status history records:", df_status.count())
print("Weather forecast records:", df_weather.count())


In [0]:

# Inspect the schema of each dataset to identify column names and data types

print("=== STATION INFORMATION ===")
df_station_info.printSchema()

print("=== STATION STATUS HISTORY ===")
df_status.printSchema()

print("=== WEATHER FORECAST HISTORY ===")
df_weather.printSchema()


## Data Cleaning and Validation

### Clean Station Status Records

Prepare the historical station status dataset for feature engineering by retaining observations that meet the following quality requirements:

- The station observation is marked as `Fresh`.
- The `station_id` is not null.
- The `station_reported_at` timestamp is not null.
- The number of available bikes is nonnegative.
- The number of available docks is nonnegative.

These checks help ensure that subsequent features and shortage prediction labels are generated from valid station observations.

In [0]:

from pyspark.sql import functions as F

# Retain fresh observations with valid station IDs, timestamps, and availability counts
df_status_clean = df_status.filter(
    (F.col("freshness_status") == "Fresh") &
    F.col("station_id").isNotNull() &
    F.col("station_reported_at").isNotNull() &
    (F.col("num_bikes_available") >= 0) &
    (F.col("num_docks_available") >= 0)
)

# Compare record counts before and after cleaning
print("Original records:", df_status.count())
print("Clean records:", df_status_clean.count())
print("Excluded records:", df_status.count() - df_status_clean.count())


### Recover Historical Service-Status Flags

Earlier station status records may contain missing values for the following service-status indicators:

- `is_installed`: Indicates whether the station is installed.
- `is_renting`: Indicates whether the station allows bike rentals.
- `is_returning`: Indicates whether the station accepts bike returns.

For observations collected before **2026-10-27 20:39:33.488**, missing values in these columns are assigned a value of `1`, based on the filtering conditions used in the earlier data ingestion process.

This correction applies only to historical records before the transition timestamp. Missing values in newer observations remain unchanged.

The original Delta table is not modified. These adjustments are applied only to the cleaned DataFrame used for feature engineering.

In [0]:

# Define the timestamp separating historical and newer observations
transition_time = "2026-10-27 20:39:33.488"

# Recover missing service-status flags for historical observations only
df_status_clean = (
    df_status_clean
    .withColumn(
        "is_installed",
        F.when(
            (F.col("collected_at") < F.to_timestamp(F.lit(transition_time))) &
            F.col("is_installed").isNull(),
            F.lit(1)
        ).otherwise(F.col("is_installed"))
    )
    .withColumn(
        "is_renting",
        F.when(
            (F.col("collected_at") < F.to_timestamp(F.lit(transition_time))) &
            F.col("is_renting").isNull(),
            F.lit(1)
        ).otherwise(F.col("is_renting"))
    )
    .withColumn(
        "is_returning",
        F.when(
            (F.col("collected_at") < F.to_timestamp(F.lit(transition_time))) &
            F.col("is_returning").isNull(),
            F.lit(1)
        ).otherwise(F.col("is_returning"))
    )
)


## Station Metadata Integration

### Join Station Status with Station Information

Combine the cleaned station status records with station metadata using the common `station_id` column.

This integration adds station-level information, including station name, geographic location, and capacity, to the historical availability observations.

The resulting dataset provides the foundation for calculating station availability ratios and other predictive features.

In [0]:

# Join cleaned station status records with station metadata using station_id
df_station_features = df_status_clean.join(
    df_station_info.select(
        "station_id",
        "name",
        "capacity",
        "lat",
        "lon"
    ),
    on="station_id",
    how="inner"
)

# Exclude stations with missing, zero, or negative capacity
df_station_features = df_station_features.filter(
    F.col("capacity") > 0
)

# Preview the joined dataset to verify station metadata and availability
display(
    df_station_features.select(
        "station_id",
        "name",
        "capacity",
        "num_bikes_available",
        "num_docks_available",
        "station_reported_at"
    ).limit(10)
)


## Feature Engineering

### Calculate Station Availability Ratios

Calculate the proportion of available bikes and docks relative to each station's total capacity.

Two features are generated:

- `bike_availability_ratio`: Number of available bikes divided by station capacity.
- `dock_availability_ratio`: Number of available docks divided by station capacity.

These features represent the relative availability of bikes and docks across stations of different sizes.

For example, a station with 5 available bikes and a capacity of 20 has a bike availability ratio of 0.25 (25%).

These ratios help the machine learning model identify station availability patterns and potential shortage conditions.

In [0]:

# Calculate bike and dock availability as proportions of station capacity
df_station_features = (
    df_station_features
    .withColumn(
        "bike_availability_ratio",
        F.col("num_bikes_available") / F.col("capacity")
    )
    .withColumn(
        "dock_availability_ratio",
        F.col("num_docks_available") / F.col("capacity")
    )
)

# Preview station availability ratios rounded to three decimal places
display(
    df_station_features.select(
        "name",
        "capacity",
        "num_bikes_available",
        "num_docks_available",
        F.round("bike_availability_ratio", 3).alias("bike_ratio"),
        F.round("dock_availability_ratio", 3).alias("dock_ratio")
    ).limit(10)
)


### Create Time-Based Features

Generate time-based features from station observation timestamps to capture daily and weekly Citi Bike usage patterns.

The following features are created:

- `station_time_nyc`: Converts the station observation timestamp from UTC to New York local time.
- `hour_of_day`: Extracts the hour of observation (0–23).
- `day_of_week`: Represents the day of the week, where Monday = 1 and Sunday = 7.
- `is_weekend`: Binary indicator where 1 represents Saturday or Sunday and 0 represents a weekday.

These features help the machine learning model identify temporal patterns in bike and dock availability, including differences between weekdays and weekends.

New York local time is used to account for the city's time zone and daylight saving time.

In [0]:

# Convert UTC station observation timestamps to New York local time
df_station_features = df_station_features.withColumn(
    "station_time_nyc",
    F.from_utc_timestamp(
        F.col("station_reported_at"),
        "America/New_York"
    )
)

# Extract hour, day of week, and weekend indicators
df_station_features = (
    df_station_features
    .withColumn("hour_of_day", F.hour("station_time_nyc"))
    .withColumn(
        "day_of_week",
        (F.dayofweek("station_time_nyc") + 5) % 7 + 1
    )
    .withColumn(
        "is_weekend",
        F.when(F.col("day_of_week").isin(6, 7), 1).otherwise(0)
    )
)

# Preview the generated time-based features
display(
    df_station_features.select(
        "name",
        "station_time_nyc",
        "hour_of_day",
        "day_of_week",
        "is_weekend"
    ).limit(10)
)


### Calculate Changes in Bike and Dock Availability

Calculate the change in available bikes and docks between consecutive observations for each Citi Bike station.

The observations are ordered chronologically within each station to ensure that changes are calculated using the correct historical records.

Two features are generated:

- `bike_change`: Difference in available bikes between the current and previous observation.
- `dock_change`: Difference in available docks between the current and previous observation.

A positive value indicates an increase in availability, while a negative value indicates a decrease.

These features help identify short-term changes in station activity that may indicate an approaching bike or dock shortage.

In [0]:

from pyspark.sql.window import Window

# Order observations chronologically for each station
station_window = Window.partitionBy("station_id").orderBy(
    "collected_at",
    "station_reported_at"
)

# Retrieve bike counts, dock counts, and timestamps from previous observations
df_station_features = (
    df_station_features
    .withColumn(
        "previous_bikes",
        F.lag("num_bikes_available").over(station_window)
    )
    .withColumn(
        "previous_docks",
        F.lag("num_docks_available").over(station_window)
    )
    .withColumn(
        "previous_collected_at",
        F.lag("collected_at").over(station_window)
    )
)

# Calculate elapsed time and availability changes for observations within 10 minutes
df_station_features = (
    df_station_features
    .withColumn(
        "minutes_since_previous",
        (
            F.unix_timestamp("collected_at") -
            F.unix_timestamp("previous_collected_at")
        ) / 60.0
    )
    .withColumn(
        "bike_change",
        F.when(
            (F.col("minutes_since_previous") > 0) &
            (F.col("minutes_since_previous") <= 10),
            F.col("num_bikes_available") - F.col("previous_bikes")
        )
    )
    .withColumn(
        "dock_change",
        F.when(
            (F.col("minutes_since_previous") > 0) &
            (F.col("minutes_since_previous") <= 10),
            F.col("num_docks_available") - F.col("previous_docks")
        )
    )
)

# Preview valid changes in bike and dock availability
display(
    df_station_features.select(
        "name",
        "collected_at",
        "num_bikes_available",
        "previous_bikes",
        "bike_change",
        "dock_change",
        F.round("minutes_since_previous", 2).alias("minutes_elapsed")
    )
    .filter(F.col("bike_change").isNotNull())
    .limit(10)
)


### Calculate 15-Minute Bike Availability Trend

Calculate the change in available bikes over approximately 15 minutes to identify short-term station availability trends.

The calculation compares the current bike count with the count recorded three observations earlier.

The resulting feature is:

- `bike_change_15min`: Difference between current bike availability and bike availability approximately 15 minutes earlier.

**Interpretation:**

- **Negative value:** Bike availability has decreased.
- **Positive value:** Bike availability has increased.
- **Zero:** Bike availability has remained unchanged.

To account for irregular data collection intervals, only observations separated by **12 to 18 minutes** are considered valid.

Observations outside this time window produce a null value rather than a potentially misleading trend.

This feature helps the machine learning model identify declining bike availability that may indicate an approaching shortage.

In [0]:

# Retrieve bike counts and timestamps from three observations earlier
df_station_features = (
    df_station_features
    .withColumn(
        "bikes_3_observations_ago",
        F.lag("num_bikes_available", 3).over(station_window)
    )
    .withColumn(
        "time_3_observations_ago",
        F.lag("collected_at", 3).over(station_window)
    )
)

# Calculate elapsed time between the current and earlier observations
df_station_features = df_station_features.withColumn(
    "elapsed_15min",
    (
        F.unix_timestamp("collected_at") -
        F.unix_timestamp("time_3_observations_ago")
    ) / 60.0
)

# Calculate bike availability changes for valid 12–18 minute intervals
df_station_features = df_station_features.withColumn(
    "bike_change_15min",
    F.when(
        F.col("elapsed_15min").between(12, 18),
        F.col("num_bikes_available") -
        F.col("bikes_3_observations_ago")
    )
)

# Preview valid 15-minute bike availability trends
display(
    df_station_features.select(
        "name",
        "num_bikes_available",
        "bikes_3_observations_ago",
        "bike_change_15min",
        F.round("elapsed_15min", 2).alias("minutes_elapsed")
    )
    .filter(F.col("bike_change_15min").isNotNull())
    .limit(10)
)


### Calculate 15-Minute Dock Availability Trend

Calculate the change in available docks over approximately 15 minutes to identify short-term station availability trends.

The calculation compares the current dock count with the count recorded three observations earlier.

The resulting feature is:

- `dock_change_15min`: Difference between current dock availability and dock availability approximately 15 minutes earlier.

**Interpretation:**

- **Negative value:** The number of available docks has decreased.
- **Positive value:** The number of available docks has increased.
- **Zero:** The number of available docks has remained unchanged.

To account for irregular data collection intervals, only observations separated by **12 to 18 minutes** are considered valid.

Observations outside this time window produce a null value rather than a potentially misleading trend.

This feature helps the machine learning model identify declining dock availability that may indicate an approaching dock shortage.

In [0]:

# Retrieve dock availability from three observations earlier
df_station_features = df_station_features.withColumn(
    "docks_3_observations_ago",
    F.lag("num_docks_available", 3).over(station_window)
)

# Calculate dock availability changes for valid 12–18 minute intervals
df_station_features = df_station_features.withColumn(
    "dock_change_15min",
    F.when(
        F.col("elapsed_15min").between(12, 18),
        F.col("num_docks_available") -
        F.col("docks_3_observations_ago")
    )
)

# Preview valid 15-minute dock availability trends
display(
    df_station_features.select(
        "name",
        "num_docks_available",
        "docks_3_observations_ago",
        "dock_change_15min",
        F.round("elapsed_15min", 2).alias("minutes_elapsed")
    )
    .filter(F.col("dock_change_15min").isNotNull())
    .limit(10)
)


## Shortage Label Generation

### Define Bike and Dock Shortage Conditions

Define the conditions used to identify bike and dock shortages at Citi Bike stations.

According to the project proposal, two shortage conditions are established:

- **Bike shortage:** A station has zero available bikes (`num_bikes_available == 0`).
- **Dock shortage:** A station has zero available docks (`num_docks_available == 0`).

Two binary indicators are generated for the current observation:

- `is_bike_shortage_now`: 1 indicates a bike shortage; 0 indicates no bike shortage.
- `is_dock_shortage_now`: 1 indicates a dock shortage; 0 indicates no dock shortage.

These indicators describe the station's current availability status.

The subsequent feature engineering process will use future station observations to generate prediction labels for shortages approximately 60 minutes ahead.

These labels will support the training and evaluation of machine learning classification models.

In [0]:

# Create binary indicators for current bike and dock shortages
df_station_features = (
    df_station_features
    .withColumn(
        "is_bike_shortage_now",
        F.when(F.col("num_bikes_available") == 0, 1).otherwise(0)
    )
    .withColumn(
        "is_dock_shortage_now",
        F.when(F.col("num_docks_available") == 0, 1).otherwise(0)
    )
)

# Preview current shortage indicators and station availability
display(
    df_station_features.select(
        "name",
        "num_bikes_available",
        "num_docks_available",
        "is_bike_shortage_now",
        "is_dock_shortage_now"
    ).limit(10)
)


### Calculate the 60-Minute Target Timestamp

Calculate the target timestamp exactly 60 minutes after each station observation to support future bike and dock shortage prediction.

For each observation, the following column is generated:

- `target_time_60min`: The timestamp 60 minutes after the current `collected_at` timestamp.

This timestamp serves as the reference point for locating future station observations used to generate shortage prediction labels.

Actual timestamps are used instead of assuming a fixed number of observations per hour because data collection intervals may vary or contain gaps.

The target timestamp identifies the desired prediction horizon; the corresponding future observation will be matched in the next section.

In [0]:

# Calculate the target timestamp 60 minutes after each station observation
df_station_features = df_station_features.withColumn(
    "target_time_60min",
    F.expr("collected_at + INTERVAL 60 MINUTES")
)

# Preview the current observation and its 60-minute target timestamp
display(
    df_station_features.select(
        "name",
        "collected_at",
        "target_time_60min"
    ).limit(10)
)


### Match Future Station Observations

Match each station observation with a future observation approximately 60 minutes later to support the generation of shortage prediction labels.

The matching process follows these conditions:

- **Station matching:** Current and future observations must belong to the same `station_id`.
- **Time tolerance:** The future observation must fall within 2.5 minutes before or after `target_time_60min`.
- **Closest observation:** If multiple future observations satisfy the time tolerance, select the observation closest to the target timestamp.
- **Missing observations:** If no qualifying future observation exists, leave the observation unmatched.

This approach accounts for irregular data collection intervals and missing observations rather than assuming that a fixed number of records represents exactly 60 minutes.

The matched future observations will be used to generate bike and dock shortage prediction labels.

In [0]:

# Prepare current station observations and their 60-minute target timestamps
current = df_station_features.alias("current")

# Select future observations from the cleaned station status history
future = df_status_clean.select(
    "station_id",
    F.col("collected_at").alias("future_collected_at"),
    F.col("num_bikes_available").alias("future_bikes"),
    F.col("num_docks_available").alias("future_docks"),
    F.col("is_installed").alias("future_is_installed"),
    F.col("is_renting").alias("future_is_renting"),
    F.col("is_returning").alias("future_is_returning")
).alias("future")

# Match observations from the same station within 150 seconds of the target time
matched = current.join(
    future,
    (F.col("current.station_id") == F.col("future.station_id")) &
    (
        F.abs(
            F.unix_timestamp("future_collected_at") -
            F.unix_timestamp("target_time_60min")
        ) <= 150
    ),
    "left"
)

# Rank matching observations by their distance from the target timestamp
match_window = Window.partitionBy(
    F.col("current.station_id"),
    F.col("current.collected_at")
).orderBy(
    F.abs(
        F.unix_timestamp("future_collected_at") -
        F.unix_timestamp("target_time_60min")
    ).asc_nulls_last()
)

# Retain the closest future observation for each station and timestamp
df_future_matched = (
    matched
    .withColumn("match_rank", F.row_number().over(match_window))
    .filter(F.col("match_rank") == 1)
)

# Preview matched future observations and their availability counts
display(
    df_future_matched.select(
        F.col("current.station_id").alias("station_id"),
        F.col("current.collected_at").alias("collected_at"),
        "target_time_60min",
        "future_collected_at",
        "future_bikes",
        "future_docks"
    ).limit(10)
)


### Create 60-Minute Shortage Labels

Generate binary prediction labels using the future station observations matched in the previous section.

Two target labels are created:

- **Bike shortage label:** A value of `1` indicates that the matched future observation has zero available bikes; otherwise, the label is `0`.
- **Dock shortage label:** A value of `1` indicates that the matched future observation has zero available docks; otherwise, the label is `0`.

**Missing Future Observations**

If no qualifying future observation is found within the specified time tolerance, the corresponding prediction labels remain null.

These records are not automatically classified as having no shortage because their future availability is unknown.

**Modeling Objective**

The generated labels will be used to train and evaluate machine learning classification models that predict bike and dock shortages approximately 60 minutes in advance.

To prevent data leakage, future availability values are used only to generate target labels, not as input features for prediction.

In [0]:

# Generate 60-minute bike and dock shortage labels for operational stations
df_labeled = (
    df_future_matched
    .withColumn(
        "bike_shortage_60min",
        F.when(
            (F.col("is_installed") == 1) &
            (F.col("is_renting") == 1) &
            (F.col("future_is_installed") == 1) &
            (F.col("future_is_renting") == 1) &
            F.col("future_bikes").isNotNull(),
            F.when(F.col("future_bikes") == 0, 1).otherwise(0)
        )
    )
    .withColumn(
        "dock_shortage_60min",
        F.when(
            (F.col("is_installed") == 1) &
            (F.col("is_returning") == 1) &
            (F.col("future_is_installed") == 1) &
            (F.col("future_is_returning") == 1) &
            F.col("future_docks").isNotNull(),
            F.when(F.col("future_docks") == 0, 1).otherwise(0)
        )
    )
)

# Preview future availability and generated shortage labels
display(
    df_labeled.select(
        F.col("current.station_id").alias("station_id"),
        "future_bikes",
        "future_docks",
        "bike_shortage_60min",
        "dock_shortage_60min"
    ).limit(10)
)


### Validate 60-Minute Shortage Label Coverage

Evaluate the completeness of the generated shortage prediction labels by identifying observations with valid labels and those with missing labels.

The validation examines:

- **Total observations:** Number of station observations processed for shortage labeling.
- **Valid bike shortage labels:** Observations with a non-null `bike_shortage_60min` value.
- **Valid dock shortage labels:** Observations with a non-null `dock_shortage_60min` value.
- **Missing labels:** Observations where the corresponding shortage label could not be generated.

Missing labels may result from unavailable future observations or unmet station operational-status requirements.

This validation helps assess whether sufficient labeled observations are available for supervised machine learning.

Observations without valid target labels will be excluded from the corresponding supervised training dataset.

In [0]:

# Summarize the completeness of 60-minute shortage prediction labels
df_label_summary = df_labeled.agg(
    F.count("*").alias("total_observations"),
    F.count("bike_shortage_60min").alias("labeled_observations"),
    F.sum(
        F.when(F.col("bike_shortage_60min").isNull(), 1).otherwise(0)
    ).alias("unlabeled_observations"),
    F.sum("bike_shortage_60min").alias("bike_shortages"),
    F.sum("dock_shortage_60min").alias("dock_shortages")
)

# Display label coverage and shortage counts
display(df_label_summary)


## Modeling Dataset Preparation

### Prepare Labeled Observations

Prepare the feature-engineered dataset for supervised machine learning by retaining observations with valid 60-minute shortage prediction labels.

The preparation process includes:

- Retaining observations with non-null `bike_shortage_60min` labels.
- Retaining observations with non-null `dock_shortage_60min` labels.
- Excluding observations without valid future shortage labels.
- Preserving the engineered features and station information needed for subsequent modeling.

The resulting dataset contains observations with valid target labels for both bike and dock shortage prediction.

This preliminary modeling dataset will undergo additional schema cleanup and validation before being saved as a Delta table.

In [0]:

# Retain observations with valid 60-minute bike and dock shortage labels
df_training_ready = df_labeled.filter(
    F.col("bike_shortage_60min").isNotNull() &
    F.col("dock_shortage_60min").isNotNull()
)

# Verify the number of observations available for model training
print("Training-ready observations:", df_training_ready.count())


### Clean the Training Dataset Schema

Clean the preliminary modeling dataset by removing duplicate column names introduced during the future-observation join.

The cleanup process includes:

- Removing the duplicate `station_id` column associated with the future observation.
- Retaining the original `station_id` from the current observation.
- Checking the dataset for remaining duplicate column names.
- Verifying the total number of columns after cleanup.

This process helps maintain a consistent dataset schema and prevents column-name conflicts when saving the data as a Delta table.

The resulting dataset will undergo further validation before being used for machine learning.

In [0]:

# Remove the duplicate station ID from the future observation
df_training_ready = df_training_ready.drop(
    F.col("future.station_id")
)

# Identify any remaining duplicate column names
column_names = df_training_ready.columns

duplicates = [
    name for name in set(column_names)
    if column_names.count(name) > 1
]

# Display schema validation results
print("Duplicate column names:", duplicates)
print("Total columns:", len(column_names))


### Preliminary Feature Dataset Preparation

The preliminary feature-engineered Citi Bike dataset (`df_training_ready`) contains historical station observations, engineered availability features, and 60-minute bike and dock shortage prediction labels.

This dataset will be inspected and validated before preparing the final machine learning dataset.

**Preliminary Dataset Considerations**

Additional preparation is required before the dataset is ready for machine learning.

In particular:

- Future observation columns and temporary matching fields must be excluded from model input features to prevent data leakage.
- The 60-minute bike and dock shortage prediction labels must be retained as supervised learning targets.
- Station operational-status fields and the final dataset schema must be reviewed.

After these checks and cleanup operations, the final modeling dataset (`df_modeling`) will be saved as a Delta table.

**Final Delta Table Destination**

`workspace.citibike.station_features_60min`

This table will provide a persistent dataset for subsequent machine learning model training and evaluation.

### Verify Preliminary Feature Dataset

Inspect the preliminary feature-engineered DataFrame (`df_training_ready`) before saving the final modeling dataset.

The validation includes:

- Checking the total number of observations with valid 60-minute shortage labels.
- Verifying the number of columns in the preliminary dataset.
- Previewing station availability features and shortage prediction labels.
- Confirming that the dataset is available for further inspection and cleanup.

This validation uses the in-memory Spark DataFrame rather than reading an existing Delta table.

Future-related columns will be reviewed and removed before the final modeling dataset is saved.

In [0]:

# Load the saved feature-engineered Delta table
df_saved = spark.table(
    "workspace.citibike.station_features_60min"
)

# Verify the number of observations and columns
print("Total observations:", df_saved.count())
print("Total columns:", len(df_saved.columns))

# Preview station availability features and 60-minute shortage labels
display(
    df_saved.select(
        "name",
        "num_bikes_available",
        "num_docks_available",
        "bike_change_15min",
        "dock_change_15min",
        "bike_shortage_60min",
        "dock_shortage_60min"
    ).limit(10)
)


### Identify Future-Related Columns

Inspect the feature-engineered dataset to identify columns containing future observation information, target timestamps, and shortage prediction labels.

This inspection helps distinguish between:

- **Future observation data:** Information collected after the current observation, which must not be used as model input features.
- **Target timestamps:** Timestamps used to locate observations approximately 60 minutes into the future.
- **Prediction labels:** The `bike_shortage_60min` and `dock_shortage_60min` columns, which serve as supervised learning targets.

Identifying these columns is important for preventing data leakage and ensuring that the machine learning model uses only information available at prediction time.

This section performs an inspection only. No columns are removed or modified during this process.

In [0]:

# Identify future observation fields, target timestamps, and shortage labels
future_columns = [
    column for column in df_saved.columns
    if "future" in column.lower()
    or "target" in column.lower()
    or "shortage_60min" in column.lower()
]

# Display the identified columns for data leakage review
print("Future-related columns:")

for column in future_columns:
    print(column)


### Remove Future-Only Columns

Remove future observation information from the modeling dataset to prevent data leakage during machine learning training and evaluation.

The cleanup process removes:

- Future observation timestamps and availability counts.
- Future station operational-status indicators.
- The 60-minute target timestamp used for matching observations.
- Temporary ranking information used to select the closest future observation.

The following prediction labels are retained:

- `bike_shortage_60min`
- `dock_shortage_60min`

These labels represent the outcomes that the machine learning models will learn to predict.

The resulting `df_modeling` DataFrame will contain historical station information, engineered features, and the two prediction targets.

This operation modifies the working DataFrame without changing the original source tables.

In [0]:

# Identify future observation fields and temporary matching columns
columns_to_remove = [
    "target_time_60min",
    "future_collected_at",
    "future_bikes",
    "future_docks",
    "future_is_installed",
    "future_is_renting",
    "future_is_returning",
    "match_rank"
]

# Remove future-only columns while retaining the shortage prediction labels
df_modeling = df_training_ready.drop(*columns_to_remove)

# Verify the number of remaining columns and confirm removal
print("Modeling columns:", len(df_modeling.columns))
print("Remaining future-only columns:",
      [col for col in columns_to_remove if col in df_modeling.columns])


In [0]:
df_modeling.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.citibike.station_features_60min")

### Review Modeling Dataset Columns

Inspect the schema of the modeling DataFrame (`df_modeling`) after removing future-only information.

Review the remaining columns to distinguish between:

- **Predictor features:** Historical station availability, operational status, time-based features, and engineered trends available at prediction time.
- **Prediction targets:** `bike_shortage_60min` and `dock_shortage_60min`.
- **Identifiers:** Station identifiers used to distinguish individual Citi Bike stations.
- **Technical columns:** Metadata and processing fields that may not be appropriate as machine learning inputs.

Use the schema inspection to verify column names and data types before preparing the final dataset for machine learning.

This section is read-only and does not modify the modeling DataFrame.

In [0]:

# Display the modeling dataset schema to review column names and data types
df_modeling.printSchema()


### Verify Removal of Future-Matching Helper Column

The `match_rank` column was created during the process of matching current station observations with future observations.

This column is used only to identify the closest matching future observation. It is a temporary processing field and must not be included as a machine learning predictor.

The earlier modeling dataset cleanup already removes `match_rank`. This section provides an additional verification to ensure that the column is absent from `df_modeling`.

If the column is still present, the cleanup operation removes it. Otherwise, the DataFrame remains unchanged.

This verification does not modify the saved Delta table.

In [0]:

# Check whether the future-matching helper column is present
print("match_rank exists:",
      "match_rank" in df_modeling.columns)

# Ensure the helper column is removed from the modeling dataset
df_modeling = df_modeling.drop("match_rank")

# Verify the final column count and removal status
print("Remaining columns:", len(df_modeling.columns))
print("match_rank removed:",
      "match_rank" not in df_modeling.columns)


### Verify Station Service-Status Fields

Inspect the historical station status table to confirm that the operational status fields required for shortage prediction are available.

The following fields are checked:

- **`is_installed`:** Indicates whether the station is installed and operationally available.
- **`is_renting`:** Indicates whether the station is accepting bike rentals.
- **`is_returning`:** Indicates whether the station is accepting bike returns.

These fields help distinguish actual bike or dock shortages from situations where a station is temporarily unavailable or out of service.

The verification checks whether each required column exists in the historical station status table.

This is a read-only inspection. No records or columns are modified.

In [0]:

# Load the historical station status table
df_status_check = spark.table(
    "workspace.citibike.station_status_history"
)

# Define the operational-status columns required for shortage prediction
required_columns = [
    "is_installed",
    "is_renting",
    "is_returning"
]

# Verify that each required column exists in the source table
for column in required_columns:
    print(
        column,
        "Present" if column in df_status_check.columns else "Missing"
    )


In [0]:
df_modeling.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.citibike.station_features_60min")

print("Corrected modeling dataset saved successfully!")

In [0]:

# Use the validated modeling dataset for the final Delta table save
df_modeling = df_modeling_check

# Verify the final modeling dataset
print("Final modeling observations:", df_modeling.count())
print("Final modeling columns:", len(df_modeling.columns))
